# Chapter 23
## Entrainment by Excitatory Input Pulses
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

LIF_ENTRAINMENT, PLOT_F_ENTRAINMENT and PLOT_F_ENTRAINMENT_2 are closed-form
analytic maps (no ODE simulation), and WB_ENTRAINMENT_INTERVALS is a
parameter sweep building an n:1 locking diagram -- Brian2 has nothing to
add over the Python port for those. The remaining three sub-examples all
simulate the same WB neuron periodically driven by an excitatory synapse,
just at different synaptic strengths, so they share one function below.

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### WB Neuron Periodically Driven by an Excitatory Synapse

A periodic external "spike" (period T) resets a fast gate `q` to 1 (not
q += 1 -- an outright reset, via `SpikeGeneratorGroup` + a synapse), `q`
decays with `tau_dq`, and drives the usual two-state synaptic gate `s`
that couples into the neuron as `-g_syn*s*vm` (implicit zero reversal
potential, as in the book's script).

In [ ]:
def simulate_WB_periodic_input(g_syn, simulation_time, T=50 * b2.ms,
                                tau_r=0.5 * b2.ms, tau_d=2 * b2.ms,
                                tau_dq=0.2097 * b2.ms, dt=0.01 * b2.ms):
    El = -65 * b2.mV
    EK = -90 * b2.mV
    ENa = 55 * b2.mV
    gl = 0.1 * b2.msiemens
    gK = 9 * b2.msiemens
    gNa = 35 * b2.msiemens
    C = 1 * b2.ufarad

    eqs = """
    alphah = 0.35 * exp(-(vm + 58.0*mV) / (20.0*mV))/ms :Hz
    alpham = 0.1/mV * (vm + 35.0*mV) / (1.0 - exp(-0.1/mV * (vm + 35.0*mV))) /ms :Hz
    alphan = -0.05/mV * (vm + 34.0*mV) / (exp(-0.1/mV * (vm + 34.0*mV)) - 1.0)/ms :Hz

    betah = 5.0 / (exp(-0.1/mV * (vm + 28.0*mV)) + 1.0)/ms :Hz
    betam = 4.0 * exp(-(vm + 60.0*mV) / (18.0*mV))/ms :Hz
    betan = 0.625 * exp(-(vm + 44.0*mV) / (80.0*mV))/ms :Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = gNa*m**3*h*(ENa-vm) + gl*(El-vm) + gK*n**4*(EK-vm) \
        - g_syn*s*vm : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dq/dt = -q/tau_dq : 1
    ds/dt = q*(1-s)/tau_r - s/tau_d : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt,
                             namespace={"g_syn": g_syn * b2.msiemens,
                                        "tau_r": tau_r, "tau_d": tau_d, "tau_dq": tau_dq})
    neuron.vm = -65 * b2.mV
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"
    neuron.q = 0
    neuron.s = 0

    num_pulses = int(simulation_time / T) + 1
    pulse_gen = b2.SpikeGeneratorGroup(1, [0] * num_pulses,
                                        (np.arange(num_pulses) + 1) * T)
    trigger = b2.Synapses(pulse_gen, neuron, on_pre="q_post = 1")
    trigger.connect()

    st_mon = b2.StateMonitor(neuron, "vm", record=True)
    net = b2.Network(neuron, pulse_gen, trigger, st_mon)
    net.run(simulation_time)
    return st_mon


def spike_times_from_trace(t, v, threshold=-20):
    idx = np.where((v[:-1] >= threshold) & (v[1:] < threshold))[0]
    v_pre, v_post = v[idx], v[idx + 1]
    t_pre, t_post = t[idx], t[idx + 1]
    return (t_pre * (-v_post - threshold) + t_post * (threshold + v_pre)) / (v_pre - v_post)

### Figure 23.4
WB Neuron 1:1-Entrained by a Strong vs. a Weak Synapse

In [ ]:
sm_strong = simulate_WB_periodic_input(0.195, 800 * b2.ms)
sm_weak = simulate_WB_periodic_input(0.14, 800 * b2.ms)

fig, ax = plt.subplots(2, figsize=(9, 6), sharex=True)
ax[0].plot(sm_strong.t / b2.ms, sm_strong.vm[0] / b2.mV, lw=1, c="k")
ax[1].plot(sm_weak.t / b2.ms, sm_weak.vm[0] / b2.mV, lw=1, c="k")
ax[0].set_ylabel("v [mV]")
ax[1].set_ylabel("v [mV]")
ax[1].set_xlabel("t [ms]")
ax[0].set_xlim(0, 800)
plt.tight_layout()
plt.show()

### Figure 23.5
Irregular Firing at Intermediate Synaptic Strength

In [ ]:
for g_syn, t_final in [(0.180, 800), (0.145, 3200)]:
    sm = simulate_WB_periodic_input(g_syn, t_final * b2.ms)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(sm.t / b2.ms, sm.vm[0] / b2.mV, lw=1, c="k")
    ax.set_xlim(0, t_final)
    ax.set_xlabel("t [ms]")
    ax.set_ylabel("v [mV]")
    ax.set_title(f"g_syn={g_syn}")
    plt.tight_layout()
    plt.show()

### Figure 23.6
N:1 Entrainment

In [ ]:
for g_syn, t_final in [(0.170, 800), (0.15, 1600)]:
    sm = simulate_WB_periodic_input(g_syn, t_final * b2.ms)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(sm.t / b2.ms, sm.vm[0] / b2.mV, lw=1, c="k")
    ax.set_xlim(0, t_final)
    ax.set_xlabel("t [ms]")
    ax.set_ylabel("v [mV]")
    ax.set_title(f"g_syn={g_syn}")
    plt.tight_layout()
    plt.show()